# NB_bishop_ch07_figures

**Bishop Chapter 7 — Key Figures: Error Surfaces, GD Trajectories, Momentum & Batch Normalization**

This notebook generates publication-quality figures for Bishop Chapter 7: 3D loss surfaces, gradient descent and SGD trajectories, momentum comparison, and batch normalization effects.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_07/NB_bishop_ch07_figures.ipynb)

In [ ]:
# Install dependencies (Colab-safe)
# !pip install numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.colors import LogNorm
from mpl_toolkits.mplot3d import Axes3D

print(f'NumPy version: {np.__version__}')

In [ ]:
# Chart styling setup
COLORS = {'blue': '#1A3A6E', 'red': '#CD0000', 'green': '#2E7D32', 'amber': '#B5853F'}

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True, dpi=300)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=300)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

## Figure 7.1: Error Surface with Local and Global Minima

A 3D surface plot of a loss function that has multiple local minima and one global minimum.

In [ ]:
# Loss function with local and global minima
def loss_surface(x, y):
    """A function with a global minimum near (1.5, -0.5) and local minima elsewhere."""
    return (0.5 * (x**2 + y**2)
            - 2.0 * np.exp(-((x - 1.5)**2 + (y + 0.5)**2) / 0.3)   # global min
            - 1.2 * np.exp(-((x + 1.0)**2 + (y - 1.0)**2) / 0.5)   # local min
            - 0.8 * np.exp(-((x + 0.5)**2 + (y + 1.5)**2) / 0.4))  # local min

x_grid = np.linspace(-3, 3.5, 300)
y_grid = np.linspace(-3, 3, 300)
X, Y = np.meshgrid(x_grid, y_grid)
Z = loss_surface(X, Y)

fig = plt.figure(figsize=(11, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, Z, cmap='coolwarm', alpha=0.85, edgecolor='none',
                rstride=3, cstride=3, antialiased=True)

# Mark minima
minima = [(1.5, -0.5, 'Global min'), (-1.0, 1.0, 'Local min'), (-0.5, -1.5, 'Local min')]
for mx, my, label in minima:
    mz = loss_surface(mx, my)
    ax.scatter([mx], [my], [mz], color=COLORS['red'], s=60, zorder=10, depthshade=False)
    ax.text(mx, my, mz + 0.3, label, fontsize=9, ha='center', color=COLORS['red'])

ax.set_xlabel('$w_1$', fontsize=11)
ax.set_ylabel('$w_2$', fontsize=11)
ax.set_zlabel('$\mathcal{L}(w_1, w_2)$', fontsize=11)
ax.set_title('Figure 7.1: Error Surface with Local and Global Minima', fontsize=13, pad=20)
ax.view_init(elev=35, azim=-60)
ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False
fig.tight_layout()
save_fig(fig, 'fig7_1_error_surface')
plt.show()

## Figure 7.2: Gradient Descent Trajectory

2D contour plot with GD trajectory showing the characteristic zigzag pattern on an elongated loss surface.

In [ ]:
# Elongated quadratic (ill-conditioned): f(x,y) = 0.5*(ax^2 + by^2)
a, b = 1.0, 20.0  # condition number = 20

def elliptic(x, y):
    return 0.5 * (a * x**2 + b * y**2)

def elliptic_grad(x, y):
    return a * x, b * y

# Gradient descent
def run_gd(x0, y0, lr, n_steps, grad_fn):
    path = [(x0, y0)]
    x, y = x0, y0
    for _ in range(n_steps):
        gx, gy = grad_fn(x, y)
        x -= lr * gx
        y -= lr * gy
        path.append((x, y))
    return np.array(path)

path_gd = run_gd(4.0, 1.5, lr=0.08, n_steps=50, grad_fn=elliptic_grad)

# Contour grid
xg = np.linspace(-5, 5, 300)
yg = np.linspace(-2, 2, 300)
Xg, Yg = np.meshgrid(xg, yg)
Zg = elliptic(Xg, Yg)

fig, ax = plt.subplots(figsize=(10, 5))
cs = ax.contour(Xg, Yg, Zg, levels=20, cmap='coolwarm', alpha=0.6)
ax.plot(path_gd[:, 0], path_gd[:, 1], 'o-', color=COLORS['blue'],
        markersize=3, linewidth=1.2, label='GD trajectory')
ax.plot(path_gd[0, 0], path_gd[0, 1], 's', color=COLORS['amber'],
        markersize=8, label='Start', zorder=10)
ax.plot(0, 0, '*', color=COLORS['red'], markersize=14, label='Minimum', zorder=10)
ax.set_xlabel('$w_1$', fontsize=12)
ax.set_ylabel('$w_2$', fontsize=12)
ax.set_title('Figure 7.2: Gradient Descent Trajectory (Zigzag on Elongated Surface)',
             fontsize=13)
ax.legend()
fig.tight_layout()
save_fig(fig, 'fig7_2_gd_trajectory')
plt.show()

## Figure 7.3: SGD Noisy Trajectory

Same contour with stochastic gradient descent showing a noisy, erratic path.

In [ ]:
# SGD: add noise to gradient
np.random.seed(7)

def run_sgd(x0, y0, lr, n_steps, grad_fn, noise_scale=0.5):
    path = [(x0, y0)]
    x, y = x0, y0
    for _ in range(n_steps):
        gx, gy = grad_fn(x, y)
        # Add gradient noise to simulate stochastic mini-batch
        gx += np.random.normal(0, noise_scale * abs(gx) + 0.1)
        gy += np.random.normal(0, noise_scale * abs(gy) + 0.1)
        x -= lr * gx
        y -= lr * gy
        path.append((x, y))
    return np.array(path)

path_sgd = run_sgd(4.0, 1.5, lr=0.08, n_steps=50, grad_fn=elliptic_grad, noise_scale=0.6)

fig, ax = plt.subplots(figsize=(10, 5))
ax.contour(Xg, Yg, Zg, levels=20, cmap='coolwarm', alpha=0.6)
ax.plot(path_sgd[:, 0], path_sgd[:, 1], 'o-', color=COLORS['red'],
        markersize=3, linewidth=1.0, alpha=0.8, label='SGD trajectory')
ax.plot(path_sgd[0, 0], path_sgd[0, 1], 's', color=COLORS['amber'],
        markersize=8, label='Start', zorder=10)
ax.plot(0, 0, '*', color=COLORS['blue'], markersize=14, label='Minimum', zorder=10)
ax.set_xlabel('$w_1$', fontsize=12)
ax.set_ylabel('$w_2$', fontsize=12)
ax.set_title('Figure 7.3: SGD Noisy Trajectory', fontsize=13)
ax.legend()
fig.tight_layout()
save_fig(fig, 'fig7_3_sgd_noise')
plt.show()

## Figure 7.4: Effect of Momentum

Side-by-side comparison of GD without momentum (zigzag) and with momentum (smoother convergence).

In [ ]:
# GD with momentum
def run_gd_momentum(x0, y0, lr, momentum, n_steps, grad_fn):
    path = [(x0, y0)]
    x, y = x0, y0
    vx, vy = 0.0, 0.0
    for _ in range(n_steps):
        gx, gy = grad_fn(x, y)
        vx = momentum * vx - lr * gx
        vy = momentum * vy - lr * gy
        x += vx
        y += vy
        path.append((x, y))
    return np.array(path)

path_no_mom = run_gd(4.0, 1.5, lr=0.08, n_steps=50, grad_fn=elliptic_grad)
path_mom = run_gd_momentum(4.0, 1.5, lr=0.08, momentum=0.9, n_steps=50,
                           grad_fn=elliptic_grad)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax in axes:
    ax.contour(Xg, Yg, Zg, levels=20, cmap='coolwarm', alpha=0.6)
    ax.plot(0, 0, '*', color=COLORS['red'], markersize=14, zorder=10)
    ax.set_xlabel('$w_1$', fontsize=12)
    ax.set_ylabel('$w_2$', fontsize=12)

# Without momentum
axes[0].plot(path_no_mom[:, 0], path_no_mom[:, 1], 'o-', color=COLORS['blue'],
             markersize=3, linewidth=1.2, label='Without momentum')
axes[0].plot(path_no_mom[0, 0], path_no_mom[0, 1], 's', color=COLORS['amber'],
             markersize=8, zorder=10)
axes[0].set_title('GD Without Momentum (zigzag)', fontsize=12)
axes[0].legend()

# With momentum
axes[1].plot(path_mom[:, 0], path_mom[:, 1], 'o-', color=COLORS['green'],
             markersize=3, linewidth=1.2, label='With momentum ($\\beta=0.9$)')
axes[1].plot(path_mom[0, 0], path_mom[0, 1], 's', color=COLORS['amber'],
             markersize=8, zorder=10)
axes[1].set_title('GD With Momentum (smoother)', fontsize=12)
axes[1].legend()

fig.suptitle('Figure 7.4: Effect of Momentum on Gradient Descent', fontsize=13, y=1.02)
fig.tight_layout()
save_fig(fig, 'fig7_4_momentum')
plt.show()

## Figure 7.6: Effect of Batch Normalization on Training Loss

Simulated training loss curves with and without batch normalization, showing faster convergence and reduced oscillation.

In [ ]:
# Simulate realistic training curves
np.random.seed(42)
epochs = np.arange(1, 101)

# Without BN: slower convergence, higher variance, plateau
loss_no_bn = 2.5 * np.exp(-0.03 * epochs) + 0.3 + \
             0.15 * np.random.randn(100) * np.exp(-0.01 * epochs)
loss_no_bn = np.maximum(loss_no_bn, 0.25)

# With BN: faster convergence, lower variance
loss_bn = 2.5 * np.exp(-0.07 * epochs) + 0.12 + \
          0.05 * np.random.randn(100) * np.exp(-0.02 * epochs)
loss_bn = np.maximum(loss_bn, 0.10)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs, loss_no_bn, color=COLORS['red'], linewidth=1.8, alpha=0.9,
        label='Without Batch Normalization')
ax.plot(epochs, loss_bn, color=COLORS['green'], linewidth=1.8, alpha=0.9,
        label='With Batch Normalization')

# Shade the gap
ax.fill_between(epochs, loss_bn, loss_no_bn, alpha=0.08, color=COLORS['blue'])

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Training Loss', fontsize=12)
ax.set_title('Figure 7.6: Effect of Batch Normalization on Training', fontsize=13)
ax.legend()
fig.tight_layout()
save_fig(fig, 'fig7_6_batch_norm_effect')
plt.show()

In [ ]:
print('Notebook complete.')